# Chapter 2 — Python as an Economist's Calculator

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch02_economists_calculator.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 2.1 Variables and numbers

In [ ]:
gdp = 100              # GDP in some base year, in any units
growth_rate = 0.07     # 7 per cent a year, written as a decimal
gdp = gdp * (1 + growth_rate)      # year 1
gdp = gdp * (1 + growth_rate)      # year 2
print(gdp, type(gdp))
print(type(140))       # a whole number is an int

In [ ]:
print("GDP after 10 years at 7%:", 100 * 1.07 ** 10)
print("100 months =", 100 // 12, "years and", 100 % 12, "months")

### Floats are not exact

In [ ]:
print(0.1 + 0.2, 0.1 + 0.2 == 0.3)
print(round(2.5), round(3.5), round(2.675, 2))

## 2.2 Strings and f-strings

In [ ]:
fiscal_year = "2024-25"
print("India GDP, " + fiscal_year, len(fiscal_year))   # + joins strings
first_year = int(fiscal_year[:4])                      # characters 0 to 3
print(first_year + 1)

In [ ]:
x = 1234567.891
print(f"{x:.2f} | {x:,.0f} | {x:>12.1f} | {0.0619:.1%}")

## 2.3 Booleans: yes or no

In [ ]:
inflation = 3.4          # per cent
print(inflation > 6, 2 <= inflation <= 6, inflation < 2 or inflation > 6)

## 2.4 Compound growth and the `math` module

In [ ]:
for t in [10, 20, 30]:
    compound = 100 * (1 + 0.07) ** t
    simple = 100 * (1 + 0.07 * t)
    print(f"After {t} years: compound {compound:6.1f}, simple {simple:6.1f}")

In [ ]:
import math

print(math.log(2), math.exp(1), math.sqrt(2))
print(math.isclose(0.1 + 0.2, 0.3))     # the safe way to compare floats

### How long to double?

In [ ]:
import matplotlib.pyplot as plt

rates = [r / 10 for r in range(10, 151)]           # 1.0, 1.1, ..., 15.0 per cent
exact_t = [math.log(2) / math.log(1 + r / 100) for r in rates]
error70 = [70 / r - t for r, t in zip(rates, exact_t)]
error72 = [72 / r - t for r, t in zip(rates, exact_t)]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.axhline(0, color="black", linewidth=0.8)
ax.axvspan(6, 8, color="#9db7d5", alpha=0.25, label="India's usual range, 6–8%")
ax.plot(rates, error70, color="#c55a11", linewidth=2, label="Rule of 70 minus exact")
ax.plot(rates, error72, color="#1f4e79", linewidth=2, label="Rule of 72 minus exact")
ax.set_xlabel("Annual growth rate, %")
ax.set_ylabel("Error, years")
ax.set_title("How wrong are the rules of thumb for doubling time?",
             loc="left", fontsize=12)
ax.legend(frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Source: author's calculation.", fontsize=8, color="grey")
plt.show()

## 2.5 Writing your own functions

In [ ]:
def doubling_time(g):
    """Exact number of years for a quantity growing at rate g to double.

    g is a decimal: 0.07 means 7 per cent a year.
    """
    return math.log(2) / math.log(1 + g)


def cagr(start, end, years):
    """Compound annual growth rate between two values, as a decimal."""
    return (end / start) ** (1 / years) - 1


print(f"{doubling_time(0.07):.2f} years; CAGR {cagr(100, 200, 10):.2%}")

### Default arguments: real versus nominal growth

In [ ]:
def real_growth(nominal, inflation, exact=True):
    """Real growth rate from nominal growth and inflation (all decimals).

    exact=True uses (1 + nominal) / (1 + inflation) - 1;
    exact=False uses the approximation nominal - inflation.
    """
    if exact:
        return (1 + nominal) / (1 + inflation) - 1
    return nominal - inflation


for nominal, infl in [(0.08, 0.03), (0.12, 0.06), (0.25, 0.15)]:
    e = real_growth(nominal, infl)                  # uses the default
    a = real_growth(nominal, infl, exact=False)     # overrides it
    print(f"nominal {nominal:.0%}, inflation {infl:.0%}: exact {e:.2%}, "
          f"approx {a:.2%}, error {(a - e) * 100:.2f} points")

## 2.6 India's growth: the real numbers

In [ ]:
from indiadata import mospi_nas

nas = mospi_nas("GDP")             # annual GDP, latest estimate of each year
nas[["year", "revision", "current_price", "constant_price"]].tail(5)

In [ ]:
nas = nas.set_index("year")
real_2011 = nas.loc["2011-12", "constant_price"]
real_2024 = nas.loc["2024-25", "constant_price"]
print("Real GDP 2011-12 and 2024-25 (Rs crore):", real_2011, real_2024)

years = 2024 - 2011                     # 13 years from 2011-12 to 2024-25
india_cagr = cagr(real_2011, real_2024, years)
print(f"Real GDP grew {real_2024 / real_2011:.2f} times in {years} years")
print(f"CAGR of real GDP, 2011-12 to 2024-25: {india_cagr:.2%}")
print(f"Exact doubling time: {doubling_time(india_cagr):.1f} years")
print(f"Rule of 70:          {70 / (india_cagr * 100):.1f} years")
print(f"Rule of 72:          {72 / (india_cagr * 100):.1f} years")

In [ ]:
def growth_between(first, last, column="constant_price"):
    """CAGR and doubling time of a GDP column between two fiscal years."""
    t = int(last[:4]) - int(first[:4])
    g = cagr(nas.loc[first, column], nas.loc[last, column], t)
    return g, doubling_time(g)


for first, last, note in [("2011-12", "2019-20", "before COVID"),
                          ("2011-12", "2024-25", "provisional end"),
                          ("2011-12", "2025-26", "advance estimate end"),
                          ("2014-15", "2024-25", "last ten years")]:
    g, t2 = growth_between(first, last)
    print(f"{first} to {last} ({note:>20}): CAGR {g:.2%}, doubles in {t2:.1f} years")

### Nominal versus real

In [ ]:
nom_g, _ = growth_between("2011-12", "2024-25", column="current_price")
deflator_g = cagr(nas.loc["2011-12", "current_price"] / real_2011,
                  nas.loc["2024-25", "current_price"] / real_2024, years)

print(f"Nominal GDP CAGR:         {nom_g:.2%}")
print(f"GDP deflator CAGR:        {deflator_g:.2%}")
print(f"Real CAGR, exact Fisher:  {real_growth(nom_g, deflator_g):.2%}")
print(f"Real CAGR, approximation: {real_growth(nom_g, deflator_g, exact=False):.2%}")
print(f"Real CAGR, direct:        {india_cagr:.2%}")

In [ ]:
fy = list(range(2011, 2025))                     # 2011, ..., 2024
actual = [nas.loc[f"{y}-{(y + 1) % 100:02d}", "constant_price"] / 1e5 for y in fy]
trend = [real_2011 / 1e5 * (1 + india_cagr) ** (y - 2011) for y in fy]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(fy, actual, color="#1f4e79", marker="o", linewidth=2,
        label="Actual real GDP")
ax.plot(fy, trend, color="#c55a11", linestyle="--",
        label=f"Constant growth at the CAGR ({india_cagr:.1%})")
ax.set_yscale("log")
ax.set_yticks([90, 120, 150, 180])
ax.set_yticklabels(["90", "120", "150", "180"])
ax.set_ylabel("Rs lakh crore, 2011-12 prices (log scale)")
ax.set_xticks(fy[::2])
ax.set_xticklabels([f"{y}-{(y + 1) % 100:02d}" for y in fy[::2]])
ax.set_title("India's real GDP and its compound growth path", loc="left",
             fontsize=12)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Source: MoSPI, National Accounts Statistics (base "
         "2011-12), latest estimates via eSankhyiki API. 2024-25 is provisional.",
         fontsize=8, color="grey")
plt.show()

## 2.7 Rupees the Indian way: lakh and crore

In [ ]:
lakh, crore = 10 ** 5, 10 ** 7


def indian_commas(n):
    """Write a whole number with Indian digit grouping: 12,34,56,789."""
    n = int(round(n))
    sign = "-" if n < 0 else ""
    n = abs(n)
    last_three = n % 1000            # the rightmost three digits
    rest = n // 1000                 # everything to their left
    if rest == 0:
        return f"{sign}{last_three}"
    text = f"{last_three:03d}"       # keep leading zeros, e.g. 7 -> "007"
    while rest > 0:
        text = f"{rest % 100:02d}," + text     # peel off two digits at a time
        rest = rest // 100
    return sign + text.lstrip("0")


for value in [999, 100000, 12345678, -20189918]:
    print(f"{value:>10} -> {indian_commas(value)}")

In [ ]:
def format_rupees(amount):
    """Format a rupee amount in words: Rs 3.5 lakh, Rs 12.3 crore, ...

    amount is in rupees (not crore).
    """
    if amount >= 1e5 * 1e7:                      # one lakh crore or more
        return f"Rs {amount / 1e12:,.2f} lakh crore"
    if amount >= 1e7:
        return f"Rs {amount / 1e7:,.2f} crore"
    if amount >= 1e5:
        return f"Rs {amount / 1e5:.2f} lakh"
    return "Rs " + indian_commas(amount)


nominal_2024 = nas.loc["2024-25", "current_price"]       # in Rs crore
print("Nominal GDP 2024-25:", indian_commas(nominal_2024), "crore")
print("In words:", format_rupees(nominal_2024 * crore))
print(f"In Western units: Rs {nominal_2024 * crore / 1e12:.1f} trillion")

## 2.8 Money over time: EMIs and present value

In [ ]:
def emi(principal, annual_rate, years):
    """Equated monthly instalment for a loan (annual_rate as a decimal)."""
    r = annual_rate / 12
    n = years * 12
    factor = (1 + r) ** n
    return principal * r * factor / (factor - 1)


def present_value(future_amount, rate, years):
    """Value today of an amount received after `years` years."""
    return future_amount / (1 + rate) ** years


loan = 10 * lakh
monthly = emi(loan, 0.10, 7)
print(f"EMI: Rs {indian_commas(monthly)} a month")
print(f"Total interest: {format_rupees(monthly * 7 * 12 - loan)}")
print("PV of Rs 10 lakh in 10 years at 7%:",
      format_rupees(present_value(10 * lakh, 0.07, 10)))

In [ ]:
r = 0.10 / 12
pv_of_emis = 0
for month in range(1, 7 * 12 + 1):      # months 1, 2, ..., 84
    pv_of_emis = pv_of_emis + present_value(monthly, r, month)
print(f"Present value of all 84 EMIs: Rs {pv_of_emis:.2f}")
print("Equal to the loan?", math.isclose(pv_of_emis, loan))

## Key ideas

## Exercises

## Data citation

## Further reading